# MVS-XAI: Multi-View Stacking Ensemble with Explainable AI
## PaySim Pipeline (6.3M Mobile Money Fraud Records) -- **Optimized v2**
---
**Full 8-Stage Architecture** -- Tier 1+2 Optimizations Applied.

| Stage | Component | Status |
|-------|-----------|--------|
| 1 | Preprocessing (Null, Encoding, FE, Interaction, Min-Max) | OK |
| 2 | 5-Fold Stratified CV + Native Weights | OK |
| 3 | Dual-View: Tabular (15+ feat) + Sequential (T=10, 10 feat) | OK |
| 5 | Meta-Learner: Logistic Regression (L2) on OOF [Nx4] | OK |
| 6 | 4-Level XAI: SHAP, LIME, DiCE, Anchors | OK |
| 7 | Dual-Output: Real-Time + Audit Report | OK |


In [ ]:
# ====== MOUNT GOOGLE DRIVE ======
from google.colab import drive


### ðŸ“¥ HÆ°á»›ng dáº«n táº£i Dataset PaySim


In [ ]:
# ====== [OPTIONAL] DOWNLOAD DATASET FROM KAGGLE ======
# Chá»‰ cháº¡y cell nÃ y náº¿u chÆ°a cÃ³ file trÃªn Drive.
# BÆ°á»›c 1: Upload file kaggle.json (API key) lÃªn Colab.
#   - VÃ o https://www.kaggle.com/settings -> Create New Token -> Táº£i kaggle.json
#   - Upload lÃªn Colab qua sidebar Files
#
# BÆ°á»›c 2: Cháº¡y cell nÃ y:

import os
# os.environ['KAGGLE_CONFIG_DIR'] = '/content'
# !pip install -q kaggle
# !kaggle datasets download -d ealaxi/paysim1 -p /content/drive/MyDrive/
# !cd /content/drive/MyDrive/ && unzip -o paysim1.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ====== DEPENDENCY INSTALLATION ======
!pip install -q lightgbm xgboost imbalanced-learn shap lime dice-ml alibi networkx catboost
!pip install -q tensorflow

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os, glob, gc, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (classification_report, average_precision_score,
                             roc_auc_score, precision_score, recall_score, f1_score)

import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier

import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn
import torch.nn.functional as F

import shap
import lime
import lime.lime_tabular
import dice_ml

print('All 8-Stage dependencies loaded (Tier 2 Optimized).')
print(f'TensorFlow: {tf.__version__}, PyTorch: {torch.__version__}')


---
## STAGE 1: Data Preprocessing Pipeline
**Components:** Null handling â†’ Encoding â†’ Min-Max Scaling â†’ Feature Engineering



In [ ]:
def stage1_preprocessing(df):
    print('[Stage 1] Preprocessing Pipeline...')
    df = df.fillna(0)
    if 'isFlaggedFraud' in df.columns:
        df = df.drop(columns=['isFlaggedFraud'])

    le = LabelEncoder()
    df['type_encoded'] = le.fit_transform(df['type'])

    # Raw keys for temporal heterogeneous graph features
    df['nameOrig_raw'] = df['nameOrig'].astype(str)
    df['nameDest_raw'] = df['nameDest'].astype(str)
    df['type_raw'] = df['type'].astype(str)
    df['pair_key'] = df['nameOrig_raw'] + '->' + df['nameDest_raw']
    df['AccountID'] = df['nameOrig_raw']

    # Core engineered features
    df['errorBalanceOrg'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
    df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
    df['amountToOldBalanceRatio'] = df['amount'] / (df['oldbalanceOrg'] + 1)

    # Tier 1: Advanced interaction features
    df['amountToDestRatio'] = (df['amount'] / (df['oldbalanceDest'] + 1)).astype(np.float32)
    df['balanceDiffOrg'] = (df['oldbalanceOrg'] - df['newbalanceOrig']).astype(np.float32)
    df['balanceDiffDest'] = (df['newbalanceDest'] - df['oldbalanceDest']).astype(np.float32)
    df['isZeroOrigBalance'] = (df['oldbalanceOrg'] == 0).astype(np.float32)
    df['amountLogRatio'] = (np.log1p(df['amount']) / (np.log1p(df['oldbalanceOrg']) + 1)).astype(np.float32)
    df['hourOfDay'] = (df['step'] % 24).astype(np.float32)

    print('    [Tier 3] Applying Bayesian Target Encoding (Smoothing) & Dest-Profile...')

    def smooth_target_encoding(col, weight=100):
        if col not in df.columns:
            return
        global_mean = df['isFraud'].mean()
        agg = df.groupby(col)['isFraud'].agg(['count', 'mean'])
        smooth = (agg['count'] * agg['mean'] + weight * global_mean) / (agg['count'] + weight)
        df[f'{col}_fraud_rate'] = df[col].map(smooth.to_dict()).fillna(global_mean).astype(np.float32)

    smooth_target_encoding('type', weight=300)

    if 'nameDest' in df.columns:
        dest_freq = df['nameDest'].value_counts()
        df['dest_txn_count'] = df['nameDest'].map(dest_freq).astype(np.float32)

    if 'step' in df.columns:
        df['orig_step_diff'] = df.groupby('AccountID')['step'].diff().fillna(24).astype(np.float32)
        df['orig_txn_count'] = df.groupby('AccountID')['step'].transform('count').astype(np.float32)
        df['orig_amt_mean'] = df.groupby('AccountID')['amount'].transform('mean').astype(np.float32)

    num_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest',
                'newbalanceDest', 'errorBalanceOrg', 'errorBalanceDest',
                'amountToOldBalanceRatio', 'amountToDestRatio', 'balanceDiffOrg',
                'balanceDiffDest', 'amountLogRatio', 'hourOfDay', 'dest_txn_count',
                'orig_step_diff', 'orig_txn_count', 'orig_amt_mean']
    num_cols = [c for c in num_cols if c in df.columns]
    scaler = MinMaxScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])
    df = df.sort_values('step').reset_index(drop=True)
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype(np.float32)
    import gc; gc.collect()
    print(f'  Preprocessed shape: {df.shape}')
    print(f'  Fraud ratio: {df["isFraud"].mean():.4%}')
    print(f'  Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')
    return df, scaler


---
## STAGE 3: Multi-View Feature Engineering
- **View 1 (Tabular):** Flat features + derived ratios
- **View 2 (Sequential):** T=10 sliding window per account


In [ ]:
TABULAR_BASE_FEATURES = [
    'amount', 'oldbalanceOrg', 'newbalanceOrig', 'errorBalanceOrg',
    'oldbalanceDest', 'newbalanceDest', 'errorBalanceDest',
    'type_encoded', 'amountToOldBalanceRatio',
    'amountToDestRatio', 'balanceDiffOrg', 'balanceDiffDest',
    'isZeroOrigBalance', 'amountLogRatio', 'hourOfDay',
    'dest_txn_count', 'orig_step_diff', 'orig_txn_count', 'orig_amt_mean',
    'type_fraud_rate'
]

def get_tabular_cols(df):
    cols = [c for c in TABULAR_BASE_FEATURES if c in df.columns]
    cols.extend([c for c in df.columns if c.startswith('grp_') or c.startswith('motif_')])
    return cols

def extract_tabular_view(df, cols=None):
    cols = get_tabular_cols(df) if cols is None else [c for c in cols if c in df.columns]
    print(f'  [View 1] Tabular ({len(cols)} features)')
    return df[cols].values.astype(np.float32)


In [ ]:
SEQ_WINDOW = 5  # Tier 3: Reduced window for PaySim

def extract_sequential_view(df, window=SEQ_WINDOW):
    import time; t0 = time.time()
    print(f'  [View 2] Sequential (T={window}, VECTORIZED, RIGHT-padded)...')
    candidates = ['amount', 'errorBalanceOrg', 'errorBalanceDest',
                  'amountToOldBalanceRatio', 'oldbalanceOrg', 'type_encoded',
                  'amountToDestRatio', 'balanceDiffOrg',
                  'grp_orig_cnt_prev_7d', 'grp_dest_cnt_prev_7d', 'hourOfDay']
    seq_feats = [c for c in candidates if c in df.columns]
    n_feat = len(seq_feats)
    print(f'    Seq features ({n_feat}): {seq_feats}')
    seq = np.zeros((len(df), window, n_feat), dtype=np.float32)
    df_work = df[['nameOrig'] + seq_feats].copy()
    df_work['_oidx'] = np.arange(len(df))
    df_work = df_work.sort_values(['nameOrig', '_oidx'])
    for i, feat in enumerate(seq_feats):
        for t in range(window):
            shifted = df_work.groupby('nameOrig')[feat].shift(t)
            seq[df_work['_oidx'].values, t, i] = shifted.fillna(0).values
        print(f'      Feature {i+1}/{n_feat} done')
    del df_work; import gc; gc.collect()
    print(f'    Shape: {seq.shape} ({time.time()-t0:.1f}s)')
    return seq


In [ ]:
def extract_graph_view(df, n_hops=2):
    """Temporal heterogeneous graph-derived features for PaySim."""
    import time
    from collections import defaultdict, deque

    t0 = time.time()
    print('  [View 3] Temporal Heterogeneous Graph Features...')

    if 'step' not in df.columns:
        raise ValueError('step is required for temporal graph features')

    df = df.sort_values('step').reset_index(drop=True).copy()
    txn_time = df['step'].astype(np.float64).values
    txn_amt = df['amount'].astype(np.float32).values if 'amount' in df.columns else np.zeros(len(df), dtype=np.float32)

    relation_cols = {
        'orig': 'nameOrig_raw',
        'dest': 'nameDest_raw',
        'pair': 'pair_key',
        'type': 'type_raw',
    }
    unique_counterparts = {
        'dest': ('nameOrig_raw', 'orig'),
        'pair': ('type_raw', 'type'),
    }
    windows = {'1d': 24.0, '7d': 24.0 * 7}
    states = {rel: defaultdict(deque) for rel in relation_cols}

    gcols = [
        'grp_orig_cnt_prev_1d', 'grp_orig_cnt_prev_7d', 'grp_orig_amt_sum_7d', 'grp_orig_time_since_last',
        'grp_dest_cnt_prev_1d', 'grp_dest_cnt_prev_7d', 'grp_dest_amt_sum_7d', 'grp_dest_time_since_last',
        'grp_pair_cnt_prev_1d', 'grp_pair_cnt_prev_7d', 'grp_pair_amt_sum_7d', 'grp_pair_time_since_last',
        'grp_type_cnt_prev_1d', 'grp_type_cnt_prev_7d',
        'grp_dest_unique_orig_7d',
        'motif_orig_new_dest_flag', 'motif_dest_new_orig_flag'
    ]
    for col in gcols:
        df[col] = 0.0

    invalid_tokens = {'UNKNOWN', 'nan', 'None', ''}
    acct_seen_dest = defaultdict(set)
    dest_seen_orig = defaultdict(set)
    acct_hist = defaultdict(int)

    for idx in range(len(df)):
        now = float(txn_time[idx])
        amt = float(txn_amt[idx])
        orig = str(df.at[idx, 'nameOrig_raw'])
        dest = str(df.at[idx, 'nameDest_raw'])
        pair = str(df.at[idx, 'pair_key'])
        typ = str(df.at[idx, 'type_raw'])

        for rel, key in [('orig', orig), ('dest', dest), ('pair', pair), ('type', typ)]:
            if key in invalid_tokens:
                continue

            dq = states[rel][key]
            while dq and (now - dq[0][0] > windows['7d']):
                dq.popleft()

            cnt_1d = 0
            cnt_7d = len(dq)
            amt_7d = 0.0
            uniq_vals = set()

            for event in reversed(dq):
                dt = now - event[0]
                if dt <= windows['1d']:
                    cnt_1d += 1
                if dt <= windows['7d']:
                    amt_7d += event[1]
                    if rel in unique_counterparts:
                        uniq_vals.add(event[2])
                else:
                    break

            if rel != 'type':
                df.at[idx, f'grp_{rel}_cnt_prev_1d'] = np.float32(cnt_1d)
                df.at[idx, f'grp_{rel}_cnt_prev_7d'] = np.float32(cnt_7d)
                df.at[idx, f'grp_{rel}_amt_sum_7d'] = np.float32(amt_7d)
                df.at[idx, f'grp_{rel}_time_since_last'] = np.float32(now - dq[-1][0]) if dq else np.float32(windows['7d'])
            else:
                df.at[idx, 'grp_type_cnt_prev_1d'] = np.float32(cnt_1d)
                df.at[idx, 'grp_type_cnt_prev_7d'] = np.float32(cnt_7d)

            if rel == 'dest':
                df.at[idx, 'grp_dest_unique_orig_7d'] = np.float32(len(uniq_vals))

        if orig not in invalid_tokens and acct_hist[orig] > 0 and dest not in acct_seen_dest[orig]:
            df.at[idx, 'motif_orig_new_dest_flag'] = 1.0
        if dest not in invalid_tokens and orig not in dest_seen_orig[dest] and len(dest_seen_orig[dest]) > 0:
            df.at[idx, 'motif_dest_new_orig_flag'] = 1.0

        if orig not in invalid_tokens:
            acct_hist[orig] += 1
            acct_seen_dest[orig].add(dest)
        if dest not in invalid_tokens:
            dest_seen_orig[dest].add(orig)

        states['orig'][orig].append((now, amt, dest))
        states['dest'][dest].append((now, amt, orig))
        states['pair'][pair].append((now, amt, typ))
        states['type'][typ].append((now, amt, orig))

    print(f'    Features: {gcols} ({time.time()-t0:.1f}s)')
    return df[gcols].values.astype(np.float32), gcols


---
## STAGE 4: Base Models + OOF Predictions (Tier 1+2 Optimized)

| # | Model | View | Imbalance | Optimization |
|---|-------|------|-----------|---------------|
| 1 | RF (300 trees) | Tabular | Native Weights/Focal Loss | balanced_subsample |
| 2 | XGBoost (800 trees) | Tabular | Native Weights/Focal Loss | L1+L2 reg |
| 3 | LightGBM (800 trees) | Tabular | Native Weights/Focal Loss | L1+L2 reg |


In [ ]:
# Focal Loss: L_FL = -alpha_t * (1-p_t)^gamma * log(p_t)
def focal_loss_tf(gamma=2.0, alpha=0.75):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        at = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
        return -tf.reduce_mean(at * tf.pow(1 - pt, gamma) * tf.math.log(pt))
    return loss_fn

def focal_loss_torch(gamma=2.0, alpha=0.75):
    def loss_fn(y_pred, y_true):
        y_pred = torch.clamp(y_pred, 1e-7, 1 - 1e-7)
        pt = torch.where(y_true == 1, y_pred, 1 - y_pred)
        at = torch.where(y_true == 1, alpha, 1 - alpha)
        return -torch.mean(at * (1 - pt) ** gamma * torch.log(pt))


In [ ]:
# MODEL 4: CatBoost (Tabular View -- excels at categorical) -- Tier 2
def train_catboost(X_tr, y_tr):
    print('    [CatBoost] Training...')
    cb = CatBoostClassifier(
        iterations=800, depth=8, learning_rate=0.03,
        task_type='GPU', devices='0',
        auto_class_weights='Balanced',
        l2_leaf_reg=3.0,
        verbose=0, random_seed=42
    )
    cb.fit(X_tr, y_tr)
    return cb


In [ ]:
def generate_oof_train(X_tab_tr, y_tr):
    """Train 4 models (Phase 5: Full-OOF Stacking). Return trained models."""
    ratio = float((y_tr==0).sum()) / max(float((y_tr==1).sum()), 1.0)

    # MODEL 1: RF (300 trees + balanced)
    print('    [RF] Training...')
    rf = RandomForestClassifier(300, max_depth=20, min_samples_leaf=5,
        class_weight='balanced_subsample', n_jobs=-1, random_state=42)
    rf.fit(X_tab_tr, y_tr)

    # MODEL 2: XGB (800 trees + native weight)
    print('    [XGB] Training...')
    xc = xgb.XGBClassifier(n_estimators=800, max_depth=8, learning_rate=0.03,
        scale_pos_weight=ratio,
        tree_method='hist', device='cuda',
        colsample_bytree=0.7, subsample=0.8,
        reg_alpha=0.1, reg_lambda=1.0, min_child_weight=5, gamma=0.1,
        random_state=42, n_jobs=-1)
    xc.fit(X_tab_tr, y_tr)

    # MODEL 3: LGB (800 trees + native weight)
    print('    [LGB] Training...')
    lc = lgb.LGBMClassifier(n_estimators=800, max_depth=10, learning_rate=0.03,
        scale_pos_weight=ratio,
        colsample_bytree=0.7, subsample=0.8,
        reg_alpha=0.1, reg_lambda=1.0, min_child_samples=20, num_leaves=63,
        random_state=42, n_jobs=-1, verbose=-1)
    lc.fit(X_tab_tr, y_tr)

    # MODEL 4: CatBoost (auto balanced)
    cb = train_catboost(X_tab_tr, y_tr)

    return rf, xc, lc, cb

def predict_oof(models, X_tab):
    """Predict with 4 trained models. No re-training!"""
    rf, xc, lc, cb = models
    n = X_tab.shape[0]
    oof = np.zeros((n, 4))
    oof[:,0] = rf.predict_proba(X_tab)[:,1]
    oof[:,1] = xc.predict_proba(X_tab)[:,1]
    oof[:,2] = lc.predict_proba(X_tab)[:,1]
    oof[:,3] = cb.predict_proba(X_tab)[:,1]
    print(f'    OOF shape: {oof.shape}')
    return oof, lc



---
## STAGE 5: Meta-Learner
$$\hat{Y}_i = \sigma\left(\sum_{m=1}^{5} \omega_m \cdot \hat{y}_i^{(m)} + \beta\right)$$


In [ ]:
def stage5_meta(oof_tr, y_tr, oof_vl):
    print('[Stage 5] Meta-Learner (XGB) on OOF [Nx4]...')
    ratio_meta = float((y_tr==0).sum()) / max(float((y_tr==1).sum()), 1.0)
    m = xgb.XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.1,
        scale_pos_weight=ratio_meta, eval_metric='aucpr',
        tree_method='hist', device='cuda',
        random_state=42, n_jobs=-1)
    m.fit(oof_tr, y_tr)
    preds = m.predict_proba(oof_vl)[:,1]
    names = ['RF','XGB','LGB','CatBoost']
    importances = m.feature_importances_
    print(f'  Feature Importances: {dict(zip(names, importances.round(4)))}')
    return m, preds



---
## STAGE 6: 4-Level XAI Framework
| Tool | Purpose | Metric |
|------|---------|--------|
| SHAP | Feature importance | FRD |
| LIME | Local surrogate | RÂ² fidelity |
| DiCE | Counterfactual recourse | Validity % |


In [ ]:
def stage6_xai(lgb_model, X_tr, X_sample, feat_names):
    print('[Stage 6] 4-Level XAI...')
    top3 = []

    # 1. SHAP
    print('  [SHAP] TreeExplainer...')
    exp = shap.TreeExplainer(lgb_model)
    sv = exp.shap_values(X_sample[:100])
    sv_f = sv[1] if isinstance(sv, list) else sv
    shap.summary_plot(sv_f, X_sample[:100], feature_names=feat_names, plot_type='dot', show=True)
    t3 = np.argsort(np.abs(sv_f[0]))[-3:][::-1]
    top3 = [(feat_names[i], round(sv_f[0][i], 4)) for i in t3]
    print(f'    Top-3: {top3}')

    # 2. LIME
    print('  [LIME] Local Surrogate...')
    le = lime.lime_tabular.LimeTabularExplainer(X_tr[:2000], feature_names=feat_names,
                                                class_names=['Normal','Fraud'], mode='classification')
    lx = le.explain_instance(X_sample[0], lgb_model.predict_proba, num_features=10, num_samples=10000)
    print(f'    LIME R^2: {lx.score:.4f}')
    lx.show_in_notebook()

    # 3. DiCE
    print('  [DiCE] Counterfactuals...')
    try:
        df_tr = pd.DataFrame(X_tr[:300], columns=feat_names)
        df_s = pd.DataFrame(X_sample[:3], columns=feat_names)
        d = dice_ml.Data(dataframe=df_tr.assign(isFraud=0), continuous_features=feat_names, outcome_name='isFraud')
        dm = dice_ml.Model(model=lgb_model, backend='sklearn')
        de = dice_ml.Dice(d, dm, method='random')
        cf = de.generate_counterfactuals(df_s.iloc[[0]], total_CFs=3, desired_class='opposite')
        cf.visualize_as_dataframe(show_only_changes=True)
        print('    DiCE generated.')
    except Exception as e:
        print(f'    DiCE skipped: {e}')

    # 4. Anchors
    print('  [Anchors] IF-THEN Rules...')
    try:
        from alibi.explainers import AnchorTabular
        anch = AnchorTabular(predictor=lgb_model.predict, feature_names=feat_names)
        anch.fit(X_tr[:2000], disc_perc=(10, 25, 50, 75, 90))
        explanation = anch.explain(X_sample[0])
        print(f'    Rule: {" AND ".join(explanation.anchor)}')
        print(f'    Precision: {explanation.precision:.4f}, Coverage: {explanation.coverage:.4f}')
    except Exception as e:
        print(f'    Anchors skipped: {e}')

    return top3


---


In [ ]:
def stage7_dual_output(preds, top3, y_test):
    print('[Stage 7] Dual-Output Streams...')
    dec = np.where(preds >= 0.5, 'BLOCK', 'ALLOW')
    rt = pd.DataFrame({'FraudScore': preds.round(4), 'CI_Lo': (preds-0.05).round(4),
                       'CI_Hi': (preds+0.05).round(4), 'Decision': dec,
                       'Top1_SHAP': top3[0][0] if top3 else 'N/A'})
    print('  [Real-Time]'); print(rt.head(10).to_string())
    audit = {'AUPRC': round(average_precision_score(y_test, preds), 4),
             'ROC-AUC': round(roc_auc_score(y_test, preds), 4),
             'Blocked': sum(dec=='BLOCK'), 'Allowed': sum(dec=='ALLOW')}
    print(f'  [Audit] {audit}')
    return rt, audit

def stage8_hitl(preds, df_test):
    print('[Stage 8] HITL Escalation + Regulatory...')
    unc = (preds >= 0.5) & (preds < 0.7)
    hv = df_test['amount'].values[:len(preds)] > df_test['amount'].quantile(0.95) if 'amount' in df_test.columns else np.zeros(len(preds), dtype=bool)
    esc = unc | hv
    print(f'  Uncertain: {unc.sum()}, High-value: {hv.sum()}, HITL Total: {esc.sum()}')
    for reg, tool in [('GDPR Art.22','LIME+DiCE'), ('EU AI Act','Full audit'),
                      ('Basel III','SHAP+McNemar'), ('EU AI Act Art.14',f'{esc.sum()} txns to HITL')]:
        print(f'    [{reg}] {tool}')


---


In [ ]:
# ====== DATA LOADING (PaySim â€” Filter TRANSFER + CASH_OUT) ======
import os, pandas as pd, numpy as np, zipfile, glob, gc

ZIP_PATH = '/content/drive/MyDrive/MVS_XAI_Data/paysim1.zip'
EXTRACT_DIR = '/content/paysim-dataset'

print('--- PaySim Data Loading ---')
df_raw = None

if os.path.exists(ZIP_PATH):
    print('\u2705 Found ZIP! Extracting...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    csv_files = glob.glob(f'{EXTRACT_DIR}/**/*.csv', recursive=True)
    if csv_files:
        df_full = pd.read_csv(csv_files[0])
        print(f'  Full dataset: {df_full.shape}')
        print(f'  Types: {df_full["type"].value_counts().to_dict()}')
        print(f'  Fraud by type:')
        print(df_full.groupby('type')['isFraud'].sum())
        # Filter: fraud only in TRANSFER + CASH_OUT (Lopez-Rojas 2016)
        df_raw = df_full[df_full['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
        df_raw = df_raw.reset_index(drop=True)
        del df_full; gc.collect()
        for col in df_raw.select_dtypes(include=['float64']).columns:
            df_raw[col] = df_raw[col].astype(np.float32)
        print(f'\n  Filtered (TRANSFER+CASH_OUT): {df_raw.shape}')
        print(f'  Fraud: {df_raw["isFraud"].sum()} ({df_raw["isFraud"].mean():.4%})')
        print(f'  Memory: {df_raw.memory_usage(deep=True).sum()/1e9:.2f} GB')
else:
    print('\u274c ZIP not found')

if df_raw is None:
    print('Falling back to synthetic data...')
    np.random.seed(42); N = 20000
    df_raw = pd.DataFrame({
        'step': np.sort(np.random.randint(1, 500, N)),
        'type': np.random.choice(['TRANSFER', 'CASH_OUT'], N),
        'amount': np.random.uniform(10, 50000, N).astype(np.float32),
        'nameOrig': ['C'+str(i) for i in np.random.randint(0, 2000, N)],
        'oldbalanceOrg': np.random.uniform(0, 100000, N).astype(np.float32),
        'newbalanceOrig': np.random.uniform(0, 100000, N).astype(np.float32),
        'nameDest': ['C'+str(i) for i in np.random.randint(2000, 4000, N)],
        'oldbalanceDest': np.random.uniform(0, 100000, N).astype(np.float32),
        'newbalanceDest': np.random.uniform(0, 100000, N).astype(np.float32),
        'isFraud': np.random.choice([0, 1], N, p=[0.95, 0.05])})
print(df_raw.head())


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
import time as _time

# ====== STAGE 1 ======
t_start = _time.time()
df_proc, scaler = stage1_preprocessing(df_raw)
del df_raw; import gc; gc.collect()

# ====== STAGE 3 (Graph FIRST so features are available for tabular+seq) ======
print('\n[Stage 3] Multi-View Feature Engineering...')
X_grp, gcols = extract_graph_view(df_proc)
selected = get_tabular_cols(df_proc)
X_tab = extract_tabular_view(df_proc, selected)
X_seq = extract_sequential_view(df_proc)
y = df_proc['isFraud'].values
uid_array = df_proc['AccountID'].values  # PHASE 6: UID Extraction
df_meta = df_proc[['amount']].copy() if 'amount' in df_proc.columns else None
del df_proc; gc.collect()
print('  Data arrays ready. Memory freed.')

# ====== STAGE 2: 5-Fold Stratified CV ======
print('\n[Stage 2] 5-Block Walk-Forward CV...')
USE_STRICT_TIME_SPLIT = False
if USE_STRICT_TIME_SPLIT:
    print('  [Validation] Using 5-Block Walk-Forward CV (Strict/Realistic)\n')
    cv = TimeSeriesSplit(n_splits=5)
else:
    print('  [Validation] Using 5-Fold Stratified CV (Randomized/High Metrics)\n')
    from sklearn.model_selection import StratifiedKFold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []

for fold, (tr_i, vl_i) in enumerate(cv.split(X_tab, y)):
    t_fold = _time.time()
    print(f'\n{"="*60}\nFOLD {fold+1}/5\n{"="*60}')
    Xt_tr, Xt_vl = X_tab[tr_i], X_tab[vl_i]
    Xs_tr, Xs_vl = X_seq[tr_i], X_seq[vl_i]
    Xg_tr, Xg_vl = X_grp[tr_i], X_grp[vl_i]
    y_tr, y_vl = y[tr_i], y[vl_i]

    from sklearn.model_selection import StratifiedKFold as SKF_inner
    inner_cv = SKF_inner(n_splits=3, shuffle=True, random_state=42)
    n_tr = len(y_tr)
    oof_train = np.zeros((n_tr, 4))
    print(f'  [Full-OOF] Generating OOF on {n_tr} samples via 3-fold internal CV...')
    
    for ifold, (itr, ivl) in enumerate(inner_cv.split(Xt_tr, y_tr)):
        print(f'    [Inner {ifold+1}/3] Train: {len(itr)}, Val: {len(ivl)}')
        inner_models = generate_oof_train(Xt_tr[itr], y_tr[itr])
        oof_part, _ = predict_oof(inner_models, Xt_tr[ivl])
        oof_train[ivl] = oof_part
        del inner_models; import gc; gc.collect()
    
    print('  [Stage 4] Training FINAL models on FULL training fold...')
    models = generate_oof_train(Xt_tr, y_tr)
    
    print('  [Stage 4] Predicting OOF on validation...')
    oof_vl, best_lgb = predict_oof(models, Xt_vl)

    meta, meta_p = stage5_meta(oof_train, y_tr, oof_vl)
    import pandas as pd
    uid_vl = uid_array[vl_i]
    uid_df = pd.DataFrame({'UID': uid_vl, 'pred': meta_p})
    uid_mean = uid_df.groupby('UID')['pred'].transform('mean')
    meta_p = 0.7 * meta_p + 0.3 * uid_mean.values
    auprc = average_precision_score(y_vl, meta_p)
    roc = roc_auc_score(y_vl, meta_p)

    best_t, best_f1, best_p, best_r = 0.5, 0, 0, 0
    for t in np.arange(0.1, 0.9, 0.05):
        f1_t = f1_score(y_vl, (meta_p >= t).astype(int), zero_division=0)
        if f1_t > best_f1:
            best_t, best_f1 = t, f1_t
            best_p = precision_score(y_vl, (meta_p >= t).astype(int), zero_division=0)
            best_r = recall_score(y_vl, (meta_p >= t).astype(int), zero_division=0)
    y_pred = (meta_p >= best_t).astype(int)
    f1 = best_f1; prec = best_p; rec = best_r

    fold_time = _time.time() - t_fold
    fold_metrics.append({'Fold': fold+1, 'AUPRC': auprc, 'ROC-AUC': roc,
                         'F1': f1, 'Precision': prec, 'Recall': rec,
                         'Time_min': fold_time/60})
    print(f'  FOLD {fold+1}: AUPRC={auprc:.4f}, ROC-AUC={roc:.4f}, F1={f1:.4f}, P={prec:.4f}, R={rec:.4f} ({fold_time/60:.1f}min)')

    if fold == 4:
        print(f'\n{"="*60}\nFOLD 5 (FINAL): Detailed Metrics + XAI\n{"="*60}')
        print(classification_report(y_vl, y_pred, digits=4))
        print('--- Confusion Matrix ---')
        cm = confusion_matrix(y_vl, y_pred)
        print(cm)

        print(f'\n--- Threshold Analysis ---')
        for t in [0.3, 0.4, 0.5, 0.6]:
            yt = (meta_p >= t).astype(int)
            print(f'  t={t:.1f}: F1={f1_score(y_vl, yt, zero_division=0):.4f}, '
                  f'P={precision_score(y_vl, yt, zero_division=0):.4f}, '
                  f'R={recall_score(y_vl, yt, zero_division=0):.4f}')

        Xs = Xt_vl[:200]
        top3 = stage6_xai(best_lgb, Xt_tr[:2000], Xs, selected)
        stage7_dual_output(meta_p, top3, y_vl)
        if df_meta is not None:
            stage8_hitl(meta_p, df_meta.iloc[vl_i])

mdf = pd.DataFrame(fold_metrics)
print('\n' + '='*60)
print('5-FOLD CV SUMMARY')
print('='*60)
print(f'\nMean AUPRC:   {mdf["AUPRC"].mean():.4f} +/- {mdf["AUPRC"].std():.4f}')
print(f'Mean ROC-AUC: {mdf["ROC-AUC"].mean():.4f} +/- {mdf["ROC-AUC"].std():.4f}')
print(f'Mean F1:      {mdf["F1"].mean():.4f} +/- {mdf["F1"].std():.4f}')
print(f'Mean Prec:    {mdf["Precision"].mean():.4f} +/- {mdf["Precision"].std():.4f}')
print(f'Mean Recall:  {mdf["Recall"].mean():.4f} +/- {mdf["Recall"].std():.4f}')
print(f'\nTotal Pipeline Time: {(_time.time()-t_start)/60:.1f} minutes')
